# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=3000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-05 15:22:35,057] A new study created in memory with name: no-name-b00e72f4-a54f-45ac-89f8-33471ce3e38c
                                                                                                                                                                                                                   

[I 2026-07-05 15:22:35,222] Trial 11 finished with value: 0.895963908712824 and parameters: {'weight_class_0': 0.02311989450309244, 'weight_class_1': 0.1570543993352876, 'weight_class_2': 0.009665862343121797}. Best is trial 11 with value: 0.895963908712824.
[I 2026-07-05 15:22:35,230] Trial 2 finished with value: 0.9385700378780707 and parameters: {'weight_class_0': 0.08083819625163449, 'weight_class_1': 1.492086360034119, 'weight_class_2': 9.472588432563894}. Best is trial 2 with value: 0.9385700378780707.
[I 2026-07-05 15:22:35,231] Trial 9 finished with value: 0.8560889758950183 and parameters: {'weight_class_0': 0.03824532413044154, 'weight_class_1': 0.037990348428025525, 'weight_class_2': 0.003739925565184149}. Best is trial 2 with value: 0.9385700378780707.
[I 2026-07-05 15:22:35,238] Trial 5 finished with value: 0.785736939064703 and parameters: {'weight_class_0': 0.3753210395297456, 'weight_class_1': 0.09834779943480151, 'weight_class_2': 0.0027570735187547714}. Best is trial 

Best trial: 14. Best value: 0.949275:   1%|█                                                                                                                                     | 24/3000 [00:00<01:03, 46.84it/s]

[I 2026-07-05 15:22:35,385] Trial 13 finished with value: 0.6387573602127433 and parameters: {'weight_class_0': 0.0011493021461955154, 'weight_class_1': 3.7794056030194114, 'weight_class_2': 0.02150331895351918}. Best is trial 2 with value: 0.9385700378780707.
[I 2026-07-05 15:22:35,394] Trial 15 finished with value: 0.9472863819417698 and parameters: {'weight_class_0': 0.19286208680356037, 'weight_class_1': 4.789353901089488, 'weight_class_2': 7.8479742820727}. Best is trial 15 with value: 0.9472863819417698.
[I 2026-07-05 15:22:35,412] Trial 14 finished with value: 0.9492754112371443 and parameters: {'weight_class_0': 0.3109534868100954, 'weight_class_1': 4.429093046889, 'weight_class_2': 5.194426590398563}. Best is trial 14 with value: 0.9492754112371443.
[I 2026-07-05 15:22:35,426] Trial 16 finished with value: 0.9468775274322292 and parameters: {'weight_class_0': 0.21015167794547726, 'weight_class_1': 3.5155167258165214, 'weight_class_2': 9.420726505223355}. Best is trial 14 with 

Best trial: 14. Best value: 0.949275:   1%|█▌                                                                                                                                    | 35/3000 [00:00<00:58, 50.67it/s]

...

[I 2026-07-05 15:23:50,680] Trial 2978 finished with value: 0.949644672246098 and parameters: {'weight_class_0': 0.32298198410756873, 'weight_class_1': 3.742247827764971, 'weight_class_2': 3.678530978175103}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:50,717] Trial 2979 finished with value: 0.9497092494994875 and parameters: {'weight_class_0': 0.32462925540264764, 'weight_class_1': 3.9683777740517763, 'weight_class_2': 3.6984792747721245}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:50,736] Trial 2981 finished with value: 0.9497347186685955 and parameters: {'weight_class_0': 0.33554044843467495, 'weight_class_1': 4.126206411044075, 'weight_class_2': 3.8444044132699378}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:50,746] Trial 2980 finished with value: 0.9497009609581316 and parameters: {'weight_class_0': 0.3292192474456202, 'weight_class_1': 3.9872810830708345, 'weight_class_2': 3.6770866295724627}. Best

Best trial: 2808. Best value: 0.94974: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 2990/3000 [01:16<00:00, 27.55it/s]

[I 2026-07-05 15:23:50,868] Trial 2986 finished with value: 0.9494355563555726 and parameters: {'weight_class_0': 0.21696021465270202, 'weight_class_1': 1.9169162045445418, 'weight_class_2': 2.34377588939595}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:50,889] Trial 2985 finished with value: 0.9496704606844607 and parameters: {'weight_class_0': 0.18102994205946119, 'weight_class_1': 2.239642170459803, 'weight_class_2': 2.175045970309914}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:50,958] Trial 2987 finished with value: 0.949626037733814 and parameters: {'weight_class_0': 0.19143418425427852, 'weight_class_1': 2.3736057162333974, 'weight_class_2': 2.4638959579175213}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:50,979] Trial 2988 finished with value: 0.9495455152925424 and parameters: {'weight_class_0': 0.18857937812899514, 'weight_class_1': 1.858288521092577, 'weight_class_2': 1.816348624626746}. Best i

Best trial: 2808. Best value: 0.94974: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [01:16<00:00, 39.38it/s]

[I 2026-07-05 15:23:51,159] Trial 2993 finished with value: 0.9496772434625737 and parameters: {'weight_class_0': 0.18026106362764202, 'weight_class_1': 2.564529095039632, 'weight_class_2': 2.067348297654121}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:51,168] Trial 2991 finished with value: 0.9496347458342881 and parameters: {'weight_class_0': 0.19217855496299668, 'weight_class_1': 2.5864361401203593, 'weight_class_2': 1.8685749438380488}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:51,174] Trial 2992 finished with value: 0.9496118785010879 and parameters: {'weight_class_0': 0.1888982362300205, 'weight_class_1': 2.5546760188863034, 'weight_class_2': 2.029023564542855}. Best is trial 2808 with value: 0.9497396954017296.
[I 2026-07-05 15:23:51,181] Trial 2994 finished with value: 0.9495109450863684 and parameters: {'weight_class_0': 0.2082697549785295, 'weight_class_1': 2.1705386689061035, 'weight_class_2': 2.520778744808888}. Best 

In [8]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [9]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [10]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.0.1_xgboost_submission.csv', index=False)

In [11]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
